# Day 15: Pandas 核心 —— groupby、agg、merge、concat

> **目标**: 掌握 Pandas 最核心的三大操作：分组聚合、表关联、拼接。这是数据岗的「瑞士军刀」。
> **前置**: Day 13-14 的 DataFrame 基础
> **数据**: `../data/sales.csv` + `../data/customers.csv`

## 1. groupby —— 分组聚合的基础

`groupby` 是 Pandas 最核心的操作之一。它的工作流程：「**拆分 → 应用 → 合并**」

```
df.groupby('分组列')['聚合列'].聚合函数()
```

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sales.csv")

# 按 country 分组，求 total 的总和
country_sales = df.groupby("country")["total"].sum()
print(country_sales)
print(type(country_sales))  # <class 'pandas.core.series.Series'>

# 按 category 分组，求 quantity 的平均值
cat_qty = df.groupby("category")["quantity"].mean()
print(cat_qty)

# 多列聚合：对同一列应用多个函数 → 返回 DataFrame
multi = df.groupby("country")["total"].agg(["sum", "mean", "count"])
print(multi)

# 对多列分别聚合
multi2 = df.groupby("country").agg({
    "total": "sum",
    "quantity": "mean"
})
print(multi2)

country
China       80101
France     208165
Germany    144020
UK         534817
US         300613
Name: total, dtype: int64
<class 'pandas.core.series.Series'>
category
Accessory    2.966667
Audio        3.194030
Computer     2.918750
Mobile       2.892473
Name: quantity, dtype: float64
            sum         mean  count
country                            
China     80101  2503.156250     32
France   208165  2602.062500     80
Germany  144020  2182.121212     66
UK       534817  2714.807107    197
US       300613  2404.904000    125
          total  quantity
country                  
China     80101  3.093750
France   208165  2.937500
Germany  144020  2.727273
UK       534817  2.959391
US       300613  3.096000


## 2. agg —— 灵活的多聚合

`agg` 可以同时对一个或多个列应用多种聚合函数。

In [2]:
# 对单列应用多个聚合函数
result = df.groupby("country")["total"].agg(["sum", "mean", "std", "max", "min"])
print(result)

# 对多列应用不同聚合函数
result2 = df.groupby("country").agg({
    "total": ["sum", "mean"],
    "quantity": ["mean", "max"],
    "price": "min"
})
print(result2)

# 自定义聚合函数（lambda）
result3 = df.groupby("country")["total"].agg([
    "sum",
    "mean",
    ("range", lambda x: x.max() - x.min())  # 自定义：极差
])
print(result3)

            sum         mean          std   max  min
country                                             
China     80101  2503.156250  1842.153755  6495   99
France   208165  2602.062500  2387.795129  9995   99
Germany  144020  2182.121212  2254.833895  9995   99
UK       534817  2714.807107  2662.263070  9995   99
US       300613  2404.904000  2280.839611  9995   99
          total               quantity     price
            sum         mean      mean max   min
country                                         
China     80101  2503.156250  3.093750   5    99
France   208165  2602.062500  2.937500   5    99
Germany  144020  2182.121212  2.727273   5    99
UK       534817  2714.807107  2.959391   5    99
US       300613  2404.904000  3.096000   5    99
            sum         mean  range
country                            
China     80101  2503.156250   6396
France   208165  2602.062500   9896
Germany  144020  2182.121212   9896
UK       534817  2714.807107   9896
US       300613  2404

## 3. transform —— 分组变换（保留原 shape）

`transform` 是对每个分组做计算，但返回和原 DataFrame **等长**的结果，常用于「分组填充」和「分组标准化」。

In [3]:
# transform 返回和原 df 等长的 Series
group_mean = df.groupby("country")["total"].transform("mean")
print(group_mean.shape)   # (500,) 和 df 等长
print(group_mean.head())

# 应用：计算每个订单相对于国家平均的偏差
df["total_vs_country_avg"] = df["total"] - df.groupby("country")["total"].transform("mean")
print(df[["country", "total", "total_vs_country_avg"]].head(10))

# 应用：分组排名（每个国家内的订单额排名）
df["country_rank"] = df.groupby("country")["total"].transform("rank", ascending=False)
print(df[df["country"] == "UK"][["total", "country_rank"]].head())

# 应用：分组填充缺失值（复习 Day 14）
df.loc[df.sample(5).index, "total"] = np.nan
df["total_filled"] = df["total"].fillna(
    df.groupby("country")["total"].transform("mean")
)
print(df[df["total"].isnull()][["country", "total", "total_filled"]].head())

(500,)
0    2182.121212
1    2404.904000
2    2404.904000
3    2404.904000
4    2602.062500
Name: total, dtype: float64
   country  total  total_vs_country_avg
0  Germany   2598            415.878788
1       US     99          -2305.904000
2       US    396          -2008.904000
3       US    396          -2008.904000
4   France    495          -2107.062500
5       UK    198          -2516.807107
6       UK   6495           3780.192893
7       UK   1198          -1516.807107
8       UK   1495          -1219.807107
9  Germany    297          -1885.121212
    total  country_rank
5     198         185.5
6    6495          21.5
7    1198         121.0
8    1495         105.0
10   1797          94.5
     country  total  total_filled
196   France    NaN   2618.556962
262       UK    NaN   2733.446154
268       UK    NaN   2733.446154
328  Germany    NaN   2234.765625
349  Germany    NaN   2234.765625


## 4. merge —— SQL JOIN 的 Pandas 实现

| Pandas | SQL | 说明 |
|--------|-----|------|
| `how='inner'` | INNER JOIN | 交集，只保留匹配上的 |
| `how='left'` | LEFT JOIN | 保留左表全部，右表不匹配填 NaN |
| `how='right'` | RIGHT JOIN | 保留右表全部 |
| `how='outer'` | FULL JOIN | 并集 |

⚠️ **merge 后也必查未匹配行（NaN）** —— 和 SQL JOIN 一样。

In [4]:
customers = pd.read_csv("../data/customers.csv")
print(customers)

# INNER JOIN：只保留有订单的客户
inner = pd.merge(df, customers, on="customer_id", how="inner")
print(f"inner: {len(inner)} 行")

# LEFT JOIN：保留所有订单，客户信息不匹配填 NaN
left = pd.merge(df, customers, on="customer_id", how="left")
print(f"left: {len(left)} 行")

# 检查未匹配的客户（LEFT JOIN 后右表列为 NaN）
unmatched = left[left["name"].isnull()]
print(f"未匹配订单数: {len(unmatched)}")
print(unmatched["customer_id"].unique())

# 多列关联（on=['col1', 'col2']）
# 不同列名关联（left_on='a', right_on='b'）
# 去除重复列（suffixes=('_sales', '_cust')）

   customer_id     name    country signup_date
0         C001    Alice     France  2023-01-15
1         C002      Bob    Germany  2023-02-20
2         C003  Charlie     France  2023-03-10
3         C004    David         US  2023-04-05
4         C005      Eva         US  2023-05-12
5         C006    Frank      China  2023-06-18
6         C007    Grace    Germany  2023-07-22
7         C008    Henry         UK  2023-08-30
8         C009      Ivy      Japan  2023-09-14
9         C010     Jack     Canada  2023-10-01
10        C011     Kate  Australia  2023-11-11
inner: 500 行
left: 500 行
未匹配订单数: 0
[]


## 5. concat —— 拼接 DataFrame

| 方向 | 参数 | 效果 |
|------|------|------|
| 纵向（上下） | `axis=0` | 行增加，列取并集 |
| 横向（左右） | `axis=1` | 列增加，行取并集 |

⚠️ **纵向拼接默认按索引对齐**。如果两个 df 索引不同且没有共同列，结果会有大量 NaN。建议用 `ignore_index=True` 重置索引。

In [5]:
# 纵向拼接：把 DataFrame 分成两部分再拼回来
df1 = df.iloc[:200]
df2 = df.iloc[200:]
df_combined = pd.concat([df1, df2], axis=0, ignore_index=True)
print(df_combined.shape)

# 横向拼接：把两个相关表按行索引拼在一起
df_left = df[["order_id", "customer_id", "total"]]
df_right = df[["order_id", "country", "category"]]
df_wide = pd.concat([df_left, df_right], axis=1)
print(df_wide.head())

# 横向拼接时，如果两个 df 有重复列名，会产生 _x / _y
# 建议先 drop 重复列，或用 merge 的 on 参数

(500, 12)
  order_id customer_id   total order_id  country   category
0    O1000        C007  2598.0    O1000  Germany  Accessory
1    O1001        C004    99.0    O1001       US  Accessory
2    O1002        C005   396.0    O1002       US   Computer
3    O1003        C007   396.0    O1003       US      Audio
4    O1004        C003   495.0    O1004   France     Mobile


## 6. 分组筛选 —— filter

`filter` 保留满足条件的组（整个组保留或删除），不是逐行筛选。

In [6]:
# 保留订单数量超过 50 的国家
big_countries = df.groupby("country").filter(lambda x: len(x) > 50)
print(f"原行数: {len(df)}, 筛选后: {len(big_countries)}")
print(big_countries["country"].unique())

# 保留平均订单额超过 2000 的品类
high_cats = df.groupby("category").filter(lambda x: x["total"].mean() > 2000)
print(high_cats["category"].unique())

原行数: 500, 筛选后: 468
['Germany' 'US' 'France' 'UK']
['Accessory' 'Computer' 'Audio' 'Mobile']


## 今日要点总结

| 操作 | 核心代码 | 返回值 | 适用场景 |
|------|----------|--------|----------|
| groupby + 单列聚合 | `df.groupby('g')['x'].sum()` | Series | 简单分组统计 |
| groupby + agg | `df.groupby('g')['x'].agg(['sum','mean'])` | DataFrame | 多统计量 |
| groupby + 多列 | `df.groupby('g').agg({'x':'sum','y':'mean'})` | DataFrame | 不同列不同统计 |
| transform | `df.groupby('g')['x'].transform('mean')` | 等长Series | 分组填充/标准化 |
| merge | `pd.merge(a, b, on='k', how='left')` | DataFrame | 表关联 |
| concat | `pd.concat([a, b], axis=0)` | DataFrame | 拼接 |
| filter | `df.groupby('g').filter(lambda x: ...)` | DataFrame | 按组条件筛选 |

**核心心法**:
- groupby 后想「降维」（每组一行）→ 用 `agg`
- groupby 后想「保留原 shape」（每行一值）→ 用 `transform`
- 多表关联 → 用 `merge`（先想 INNER vs LEFT）
- 表拼接 → 用 `concat`（先确认索引/列对齐）
- merge 后也查 NaN → 和 SQL JOIN 后的 COALESCE 检查等价